# Part 2

## 1. Setup and Configuration

In [1]:
import os
import glob
import pandas as pd
from dotenv import load_dotenv
from litellm import completion

load_dotenv(os.path.join("..", ".env"))

DATA_DIR = os.path.abspath(os.path.join("..", "data"))
CSV_PATH = os.path.join(DATA_DIR, "structured", "daily_sales.csv")
TEXT_DIR = os.path.join(DATA_DIR, "unstructured")
MODEL = "anthropic/claude-sonnet-4-20250514"

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set!"
assert os.path.isfile(CSV_PATH), f"CSV not found at {CSV_PATH}"
assert os.path.isdir(TEXT_DIR), f"Text dir not found at {TEXT_DIR}"

## 2. Load Data Sources

In [2]:
df = pd.read_csv(CSV_PATH)
print(f"CSV: {len(df)} rows, columns: {list(df.columns)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Categories: {sorted(df['category'].unique())}")
print(f"Regions: {sorted(df['region'].unique())}")
print()

product_texts = {}
for filepath in sorted(glob.glob(os.path.join(TEXT_DIR, "*.txt"))):
    filename = os.path.basename(filepath)
    product_id = filename.replace("_product_page.txt", "")
    with open(filepath, "r") as f:
        product_texts[product_id] = f.read()

print(f"Product pages loaded: {len(product_texts)}")
print(f"Product IDs: {list(product_texts.keys())}")

CSV: 1000 rows, columns: ['date', 'product_id', 'product_name', 'category', 'units_sold', 'unit_price', 'total_revenue', 'region']
Date range: 2024-10-03 to 2024-12-31
Categories: ['Beauty & Personal Care', 'Books', 'Clothing', 'Electronics', 'Food & Grocery', 'Home & Kitchen', 'Office Supplies', 'Pet Supplies', 'Sports & Outdoors', 'Toys & Games']
Regions: ['Central', 'East', 'North', 'South', 'West']

Product pages loaded: 10
Product IDs: ['BEAU001', 'BOOK001', 'CLTH001', 'ELEC001', 'FOOD001', 'HOME003', 'OFFC001', 'PETS001', 'SPRT001', 'TOYS001']


## 3. Query Router

In [3]:
def route_query(question: str) -> str:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a query router for an e-commerce RAG system with two data sources:\n"
                    "- CSV: daily sales data with columns (date, product_id, product_name, category, units_sold, unit_price, total_revenue, region). Use for numerical/analytical questions about sales, revenue, quantities, rankings by sales.\n"
                    "- TEXT: product pages with descriptions, features, specs, and customer reviews. Use for questions about product details, features, customer opinions.\n\n"
                    "Classify the query into exactly one of: csv, text, both\n"
                    "Respond with ONLY one word: csv, text, or both"
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    route = response.choices[0].message.content.strip().lower()
    if route not in ("csv", "text", "both"):
        route = "both"
    return route

test_qs = [
    "What was the total revenue for Electronics in December 2024?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "Which product has the best reviews and sells well?",
]
for q in test_qs:
    print(f"  [{route_query(q)}] {q}")

  [csv] What was the total revenue for Electronics in December 2024?
  [text] What are the key features of the Wireless Bluetooth Headphones?
  [both] Which product has the best reviews and sells well?


## 4. CSV Retrieval


In [4]:
def retrieve_csv(question: str) -> str:
    schema_info = (
        f"DataFrame 'df' with {len(df)} rows.\n"
        f"Columns: {list(df.columns)}\n"
        f"dtypes: {df.dtypes.to_dict()}\n"
        f"Categories: {sorted(df['category'].unique().tolist())}\n"
        f"Regions: {sorted(df['region'].unique().tolist())}\n"
        f"Products: {sorted(df['product_name'].unique().tolist())}\n"
        f"Date range: {df['date'].min()} to {df['date'].max()}\n"
        f"Sample:\n{df.head(3).to_string()}"
    )
    
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You write Python pandas code to answer questions about sales data. "
                    "The DataFrame is already loaded as 'df'. The 'date' column is a string in YYYY-MM-DD format. "
                    "Write code that computes the answer and stores it in a variable called 'result'. "
                    "Output ONLY the Python code. No markdown, no explanations, no ```python blocks."
                ),
            },
            {
                "role": "user",
                "content": f"Schema:\n{schema_info}\n\nQuestion: {question}",
            },
        ],
        temperature=0,
    )
    
    code = response.choices[0].message.content.strip()
    if code.startswith("```"):
        code = "\n".join(code.split("\n")[1:])
    if code.endswith("```"):
        code = "\n".join(code.split("\n")[:-1])
    
    print(f"[CSV code]:\n{code}")
    
    local_vars = {"df": df, "pd": pd}
    try:
        exec(code, {"__builtins__": __builtins__}, local_vars)
        result = local_vars.get("result", "No result variable found")
        return f"[CSV Analysis]\nCode executed:\n{code}\n\nResult:\n{result}"
    except Exception as e:
        return f"[CSV Analysis Error]\nCode:\n{code}\nError: {e}"

## 5. Text Retrieval

In [5]:
from rank_bm25 import BM25Okapi
product_ids = list(product_texts.keys())
product_docs = list(product_texts.values())
tokenized_docs = [doc.lower().split() for doc in product_docs]
bm25 = BM25Okapi(tokenized_docs)

def retrieve_text(question: str, top_k: int = 3) -> str:
    tokenized_query = question.lower().split()
    scores = bm25.get_scores(tokenized_query)
    
    ranked = sorted(zip(product_ids, product_docs, scores), key=lambda x: x[2], reverse=True)
    top_results = ranked[:top_k]
    
    context_parts = []
    for pid, doc, score in top_results:
        if score > 0:
            print(f"[Text match] {pid} (score: {score:.2f})")
            context_parts.append(f"--- Product: {pid} (relevance: {score:.2f}) ---\n{doc}")
    
    if not context_parts:
        print("[Text] No strong BM25 match, returning all pages")
        for pid, doc in zip(product_ids, product_docs):
            context_parts.append(f"--- Product: {pid} ---\n{doc}")
    
    return "\n\n".join(context_parts)

## 6. Answer Generation

In [6]:
def generate_answer(question: str, context: str) -> str:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an e-commerce analytics assistant. Answer the user's question "
                    "based on the provided data context. Include specific numbers, product names, "
                    "and data points in your answer. Be thorough but concise."
                ),
            },
            {
                "role": "user",
                "content": f"## Question\n{question}\n\n## Data Context\n{context}",
            },
        ],
        temperature=0,
    )
    return response.choices[0].message.content

## 7. Full Pipeline

In [7]:
def multi_source_qa(question: str) -> dict:
    print(f"\nQuestion: {question}")
    
    route = route_query(question)
    print(f"  Route: {route}")
    
    context_parts = []
    
    if route in ("csv", "both"):
        csv_context = retrieve_csv(question)
        context_parts.append(csv_context)
    
    if route in ("text", "both"):
        text_context = retrieve_text(question)
        context_parts.append(text_context)
    
    context = "\n\n".join(context_parts)
    
    answer = generate_answer(question, context)
    
    print(f"\n Answer:\n{answer}")
    print("\n" + "=" * 80)
    
    return {
        "question": question,
        "route": route,
        "answer": answer,
    }

## 8. Run Test Questions

In [8]:
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?",
]

results = []
for q in test_questions:
    result = multi_source_qa(q)
    results.append(result)


Question: What was the total revenue for Electronics category in December 2024?
  Route: csv
[CSV code]:
df['date'] = pd.to_datetime(df['date'])
december_electronics = df[(df['date'].dt.month == 12) & (df['date'].dt.year == 2024) & (df['category'] == 'Electronics')]
result = december_electronics['total_revenue'].sum()

 Answer:
Based on the data analysis, the total revenue for the Electronics category in December 2024 was **$142,864.31**.

This figure represents the sum of all revenue generated from Electronics products during the month of December 2024.


Question: Which region had the highest sales volume?
  Route: csv
[CSV code]:
result = df.groupby('region')['units_sold'].sum().idxmax()

 Answer:
Based on the sales data analysis, **Central** region had the highest sales volume in terms of total units sold.

The analysis grouped all sales data by region and calculated the sum of units sold for each region. Among all regions in the dataset, Central region achieved the highest total 

In [9]:
output_path = os.path.join("..", "part2_results.txt")

with open(output_path, "w") as f:
    f.write("Part 2: Multi-Source RAG with Routing - Results\n")
    
    for i, r in enumerate(results, 1):
        f.write(f"Question {i}: {r['question']}\n")
        f.write(f"Route: {r['route']}\n")
        f.write(f"{r['answer']}\n")

print(f"Results saved to {os.path.abspath(output_path)}")

Results saved to /Users/satomiito/Documents/GitHub/spring-2026-a03-satomitheito/part2_results.txt
